# Analiza rynku mieszkań w Warszawie
**Eksploracyjna analiza danych (EDA) — 2000 ofert sprzedaży**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

---
## Część 1 — Wstępna eksploracja

In [ ]:
df = pd.read_csv('mieszkania_warszawa.csv')
print('=== SHAPE ===')
print(df.shape)

print('\n=== INFO ===')
df.info()

In [ ]:
print('=== DESCRIBE ===')
df.describe()

In [ ]:
print('=== BRAKUJĄCE WARTOŚCI ===')
print(df.isnull().sum())

###  Obserwacje po wstępnej eksploracji

Już z `describe()` widać kilka niepokojących sygnałów:

- **`cena_pln`** — max sięga kilkudziesięciu milionów złotych (absurdalnie drogie oferty), a min może być bliskie zera (absurdalnie tanie — błędy danych lub oszustwa). Odchylenie standardowe jest bardzo duże względem mediany.
- **`metraz_m2`** — max przekracza 300–500 m², co jest nierealistyczne dla typowego mieszkania na rynku wtórnym. Q3 powinien być ok. 70–80 m², więc tak wysokie wartości to ewidentne outliery.
- **`rok_budowy`** — min może wynosić 1800, a max 2099 — wartości niemożliwe (Warszawa nie ma budynków z 1800 r. wpisanych jako rok budowy mieszkania; rok 2099 to przyszłość). Wskazuje to na błędy danych.
- **Brak braków (`NaN`)** — wszystkie 2000 wierszy jest kompletnych, nie ma problemów z brakującymi danymi.

---
## Część 2 — Statystyki opisowe

In [ ]:
# 2.1 Statystyki cena_pln
print('=== STATYSTYKI: cena_pln ===')
print(f'Średnia:              {df["cena_pln"].mean():>15,.0f} PLN')
print(f'Mediana:              {df["cena_pln"].median():>15,.0f} PLN')
print(f'Odch. standardowe:    {df["cena_pln"].std():>15,.0f} PLN')
print(f'Skewness (skośność):  {df["cena_pln"].skew():>15.4f}')
print(f'Kurtosis (kurtozja):  {df["cena_pln"].kurt():>15.4f}')

###  Co mówi skewness?

**Skewness > 0** (skośność prawostronną) oznacza, że ogon rozkładu ciągnie się w prawo — jest dużo ofert o typowych cenach, ale istnieje mała grupa bardzo drogich mieszkań (outliery po prawej stronie), które "ciągną" średnią w górę ponad medianę. W praktyce: **mediana jest bardziej reprezentatywna niż średnia** dla tego zbioru. Im wyższy skewness, tym silniejszy efekt.

In [ ]:
# 2.2 Q1, Q3, IQR dla metraz_m2
Q1 = df['metraz_m2'].quantile(0.25)
Q3 = df['metraz_m2'].quantile(0.75)
IQR = Q3 - Q1
print('=== KWARTYLE: metraz_m2 ===')
print(f'Q1 (25. percentyl):  {Q1:.1f} m²')
print(f'Q3 (75. percentyl):  {Q3:.1f} m²')
print(f'IQR (Q3 - Q1):       {IQR:.1f} m²')

In [ ]:
# 2.3 Liczba unikalnych dzielnic i rozkład ofert
print(f'Liczba unikalnych dzielnic: {df["dzielnica"].nunique()}')
print('\nLiczba ofert według dzielnicy:')
print(df['dzielnica'].value_counts())

---
## Część 3 — Analiza pojedynczych zmiennych

In [ ]:
# 3.1 Histogram + KDE dla cena_pln i metraz_m2
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# cena_pln
sns.histplot(df['cena_pln'], bins=60, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Histogram + KDE: Cena [PLN]', fontweight='bold')
axes[0].set_xlabel('Cena (PLN)')
axes[0].set_ylabel('Liczba ofert')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# metraz_m2
sns.histplot(df['metraz_m2'], bins=60, kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('Histogram + KDE: Metraż [m²]', fontweight='bold')
axes[1].set_xlabel('Metraż (m²)')
axes[1].set_ylabel('Liczba ofert')

plt.tight_layout()
plt.savefig('hist_kde.png', bbox_inches='tight')
plt.show()
print(f'Skewness cena_pln:  {df["cena_pln"].skew():.2f}')
print(f'Skewness metraz_m2: {df["metraz_m2"].skew():.2f}')

**Obserwacja:** Obydwa rozkłady wykazują prawostronną skośność (długi ogon po prawej). Dla `cena_pln` jest to wyraźnie widoczne — większość ofert skupia się po lewej, ale kilka ekstremalnych cen rozciąga oś X w prawo. Podobnie dla `metraz_m2` — kilka mieszkań o metrażu 300–500 m² odstaje od reszty.

In [ ]:
# 3.2 Boxplot cena_pln
fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(x=df['cena_pln'], ax=ax, color='steelblue', flierprops=dict(marker='o', markersize=3, alpha=0.4))
ax.set_title('Boxplot: Cena [PLN]', fontweight='bold')
ax.set_xlabel('Cena (PLN)')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.savefig('boxplot_cena.png', bbox_inches='tight')
plt.show()

**Co widać na boxplocie?**
- **Pudełko** (IQR, 25–75 percentyl) jest wąskie i skupione po lewej stronie — większość ofert ma podobną, umiarkowaną cenę.
- **Mediana** (linia w środku pudełka) jest znacznie niżej niż średnia — potwierdzenie prawostronnej skośności.
- **Outliery** (kropki po prawej) są bardzo liczne i sięgają ekstremalnych wartości — to wstrzyknięte błędy danych (mieszkania pomnożone ×5–12).
- Brak outlierów po lewej stronie może być spowodowany skalą — tanich anomalii nie widać w tej skali.

In [ ]:
# 3.3 Countplot dzielnic (posortowany malejąco)
order = df['dzielnica'].value_counts().index
fig, ax = plt.subplots(figsize=(12, 5))
sns.countplot(data=df, y='dzielnica', order=order, palette='Blues_r', ax=ax)
ax.set_title('Liczba ofert według dzielnicy', fontweight='bold')
ax.set_xlabel('Liczba ofert')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('countplot_dzielnice.png', bbox_inches='tight')
plt.show()

---
## Część 4 — Analiza zależności

In [ ]:
# 4.1 Heatmapa korelacji
df_corr = df.copy()
df_corr['ma_balkon'] = df_corr['ma_balkon'].astype(int)
df_corr['ma_miejsce_parkingowe'] = df_corr['ma_miejsce_parkingowe'].astype(int)
corr_matrix = df_corr.select_dtypes(include='number').corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, ax=ax,
    linewidths=0.5, annot_kws={'size': 9}
)
ax.set_title('Macierz korelacji — zmienne numeryczne', fontweight='bold')
plt.tight_layout()
plt.savefig('heatmapa_korelacji.png', bbox_inches='tight')
plt.show()

print('Korelacja zmiennych z cena_pln (posortowana):')
print(corr_matrix['cena_pln'].drop('cena_pln').sort_values(ascending=False))

**Która zmienna najsilniej koreluje z ceną?** — `metraz_m2` (metraż) wykazuje zdecydowanie najsilniejszą korelację dodatnią z `cena_pln`. Jest to intuicyjne: większe mieszkanie → wyższa cena całkowita. Rok budowy i liczba pokoi mają umiarkowaną korelację.

In [ ]:
# 4.2 Scatter plot metraż vs cena
fig, ax = plt.subplots(figsize=(12, 6))
scatter = ax.scatter(
    df['metraz_m2'], df['cena_pln'],
    c=pd.Categorical(df['dzielnica']).codes,
    cmap='tab20', alpha=0.5, s=20
)
# Legenda dzielnic
dzielnice_unique = df['dzielnica'].unique()
handles = [plt.scatter([], [], c=[plt.cm.tab20(i/len(dzielnice_unique))], label=d, s=40)
           for i, d in enumerate(sorted(dzielnice_unique))]
ax.legend(handles=handles, title='Dzielnica', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.set_title('Metraż vs Cena (kolor = dzielnica)', fontweight='bold')
ax.set_xlabel('Metraż (m²)')
ax.set_ylabel('Cena (PLN)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.savefig('scatter_metraz_cena.png', bbox_inches='tight')
plt.show()

In [ ]:
# 4.3 Boxplot cena za m² według dzielnicy
df['cena_pln_per_m2'] = df['cena_pln'] / df['metraz_m2']

median_per_dzielnica = df.groupby('dzielnica')['cena_pln_per_m2'].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(13, 6))
sns.boxplot(
    data=df, x='dzielnica', y='cena_pln_per_m2',
    order=median_per_dzielnica.index,
    palette='Blues_r', ax=ax,
    flierprops=dict(marker='o', markersize=2, alpha=0.3)
)
ax.set_title('Cena za m² według dzielnicy (posortowane wg mediany)', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Cena za m² (PLN)')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('boxplot_cena_per_m2_dzielnica.png', bbox_inches='tight')
plt.show()

print('Mediana ceny za m² według dzielnicy:')
print(median_per_dzielnica.round(0))

**Która dzielnica ma najwyższe ceny za m²?** — **Śródmieście** wyraźnie dominuje, co jest zgodne z oczekiwaniami (multiplikator 1.40 w danych). Wysoko plasują się też **Wilanów** i **Mokotów**. Najtaniej jest w **Białołęce** i **Targówku**.

---
## Część 5 — Detekcja outlierów

In [ ]:
# 5.1 Trzy metody detekcji outlierów w cena_pln

# Metoda 1: IQR (1.5×)
Q1_c = df['cena_pln'].quantile(0.25)
Q3_c = df['cena_pln'].quantile(0.75)
IQR_c = Q3_c - Q1_c
lower_iqr = Q1_c - 1.5 * IQR_c
upper_iqr = Q3_c + 1.5 * IQR_c
outliers_iqr = df[(df['cena_pln'] < lower_iqr) | (df['cena_pln'] > upper_iqr)]
print(f'Metoda IQR (1.5×):         {len(outliers_iqr):>4} outlierów  [granice: {lower_iqr:,.0f} — {upper_iqr:,.0f} PLN]')

# Metoda 2: Z-score (|z| > 3)
z_scores = np.abs(stats.zscore(df['cena_pln']))
outliers_z = df[z_scores > 3]
print(f'Metoda Z-score (|z|>3):    {len(outliers_z):>4} outlierów')

# Metoda 3: Modified Z-score (|z_mod| > 3.5)
median_c = df['cena_pln'].median()
mad = np.median(np.abs(df['cena_pln'] - median_c))
modified_z = 0.6745 * (df['cena_pln'] - median_c) / mad
outliers_mod_z = df[np.abs(modified_z) > 3.5]
print(f'Metoda Modified Z (>3.5):  {len(outliers_mod_z):>4} outlierów')

In [ ]:
# 5.2 Outliery w metraz_m2 (IQR)
Q1_m = df['metraz_m2'].quantile(0.25)
Q3_m = df['metraz_m2'].quantile(0.75)
IQR_m = Q3_m - Q1_m
upper_iqr_m = Q3_m + 1.5 * IQR_m

outliers_metraz = df[df['metraz_m2'] > upper_iqr_m]
print(f'Outliery metrażu (IQR): {len(outliers_metraz)} wierszy (górna granica: {upper_iqr_m:.1f} m²)')
print('\nTop 5 największych metraży:')
print(df.nlargest(5, 'metraz_m2')[['id_oferty', 'dzielnica', 'metraz_m2', 'cena_pln']])

In [ ]:
# 5.3 Bzdurne wartości rok_budowy
bledne_daty = df[(df['rok_budowy'] < 1900) | (df['rok_budowy'] > 2026)]
print(f'Wiersze z bzdurnym rokiem budowy: {len(bledne_daty)}')
print(bledne_daty[['id_oferty', 'dzielnica', 'rok_budowy', 'cena_pln']])

---
## Część 6 — Decyzja i czyszczenie

In [ ]:
# 6.1 Usuń wiersze z nielogicznym rok_budowy
print(f'Przed czyszczeniem: {len(df)} wierszy')
df_clean = df[(df['rok_budowy'] >= 1900) & (df['rok_budowy'] <= 2026)].copy()
print(f'Po usunięciu błędnych dat: {len(df_clean)} wierszy (usunięto {len(df) - len(df_clean)})')

In [ ]:
# 6.2 Winsoryzacja cena_pln (cap na 1. i 99. percentylu)
p01 = df_clean['cena_pln'].quantile(0.01)
p99 = df_clean['cena_pln'].quantile(0.99)
df_clean['cena_pln_capped'] = df_clean['cena_pln'].clip(lower=p01, upper=p99)
print(f'Winsoryzacja cena_pln:')
print(f'  Cap dolny (p1):   {p01:,.0f} PLN')
print(f'  Cap górny (p99):  {p99:,.0f} PLN')
print(f'  Oryginalne max:   {df_clean["cena_pln"].max():,.0f} PLN')
print(f'  Po cap max:       {df_clean["cena_pln_capped"].max():,.0f} PLN')

In [ ]:
# 6.3 Transformacja logarytmiczna + porównanie skewness
df_clean['cena_pln_log'] = np.log1p(df_clean['cena_pln'])

skew_before = df_clean['cena_pln'].skew()
skew_after = df_clean['cena_pln_log'].skew()
print(f'Skewness przed transformacją (cena_pln):     {skew_before:.4f}')
print(f'Skewness po transformacji (cena_pln_log):    {skew_after:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_clean['cena_pln'], bins=60, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title(f'Rozkład cena_pln\nSkewness = {skew_before:.2f}', fontweight='bold')
axes[0].set_xlabel('Cena (PLN)')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

sns.histplot(df_clean['cena_pln_log'], bins=60, kde=True, ax=axes[1], color='seagreen')
axes[1].set_title(f'Rozkład log1p(cena_pln)\nSkewness = {skew_after:.2f}', fontweight='bold')
axes[1].set_xlabel('log1p(Cena)')

plt.suptitle('Porównanie rozkładu przed i po transformacji logarytmicznej', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('log_transform_porownanie.png', bbox_inches='tight')
plt.show()

---
## Część 7 — Wnioski

###  Najważniejsze wnioski z analizy rynku mieszkań w Warszawie

1. **Metraż to kluczowy czynnik ceny.** Zmienna `metraz_m2` wykazuje najsilniejszą korelację z `cena_pln` spośród wszystkich zmiennych numerycznych. Większe mieszkanie = proporcjonalnie wyższa cena całkowita — zależność jest niemal liniowa po odcięciu outlierów.

2. **Lokalizacja silnie różnicuje ceny za m².** Śródmieście jest zdecydowanie najdroższe (mediana ceny za m² ~19 000–20 000 PLN), a Białołęka i Targówek najtańsze (~10 000–11 000 PLN). Rozpiętość między dzielnicami sięga prawie 2×, co potwierdza kluczową rolę lokalizacji w wycenie nieruchomości.

3. **Rozkład cen jest silnie prawoskośny.** Skewness > 1 wskazuje, że większość ofert skupia się w przedziale 300 000–1 500 000 PLN, ale istnieją pojedyncze oferty ekstremalnie drogie. Mediana jest bardziej miarodajną miarą "typowej" ceny niż średnia.

4. **Zbiór zawierał celowo wstrzyknięte błędy danych.** Wykryto: 10 ofert z cenami zawyżonymi 5–12×, 10 ofert z cenami zaniżonymi do kilku procent wartości rynkowej, 5 mieszkań o metrażu 300–600 m² oraz 5 ofert z absurdalnymi latami budowy (1800, 1850, 2050, 2099). Wszystkie trzy metody detekcji outlierów (IQR, Z-score, Modified Z-score) skutecznie wykryły większość anomalii.

5. **Transformacja logarytmiczna znacznie poprawia normalność rozkładu.** Po zastosowaniu `log1p()` skośność spada z wartości powyżej 5 do zbliżonej do 0–0.5, co umożliwia zastosowanie metod statystycznych i modeli ML zakładających normalność rozkładu zmiennej docelowej.